In [ ]:
# Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

In [1]:
# Initialize Google Earth Engine API.
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm.
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")

def create_monthly_composite(month):
    # Determine the start and end dates for each month
    start_date = ee.Date.fromYMD(2018, month, 1)
    end_date = start_date.advance(1, 'month')
    
    # Filter by date, apply preprocessing, select bands, and compute median
    composite = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                 .filterDate(start_date, end_date)
                 .map(prep_sr_l8)
                 .select(['SR_B4', 'SR_B5'])
                 .median()
                 .set({
                     "system:time_start": start_date.millis(),
                     "month": month,
                     "year": 2018
                 }))
    return composite

# Create an Earth Engine list from 1 to 12
months = ee.List.sequence(1, 12)

# Map the function over the list of months to create a list of images
monthly_images = months.map(create_monthly_composite)

# Cast the resulting list of images back into an ImageCollection
ic = ee.ImageCollection.fromImages(monthly_images)

In [ ]:
# Code snippet 1: Landsat 8 export of the Red and NIR bands of 
#                 Plumas National Forest, composited for each month of 2018.

# My UDF. This will simply select the Red and NIR bands in my Landsat scene.
def get_bands(df):
    return df[["SR_B4", "SR_B5"]]


from robustraster import run

run(
    dataset=ic,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": get_bands,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_Tiles",
        "vrt": True,
        "report": True
    }
)


In [ ]:
# Execute robustraster
run(
    dataset = "User-Defined Dataset (UDD) -> e.g. path/to/rasters or ee.ImageCollection",
    dataset_config = {
    	'geometry': "Your Area of Interest",
    	'crs': "Projection EPSG",
    	'scale': "Target pixel resolution"
    },
    user_function_config = {
    	'user_function': get_bands, # Injecting your UDF here
    	'user_function_args': (),
    	'user_function_kwargs': {}
    },
    export_config = {
    	'mode': "raster",
    	'file_format': "GTiff",
    	'output_folder': "path/to/save"
    }
) 


In [ ]:
# Code Snippet 2: Using the VRTs generated from Code Snippet 1, compute the NDVI of all exported tiles.
# Load in VRT file
# Compute NDVI
# Export results to disk

# My UDF. This time, we compute the NDVI derived from the data obtained
# from Code Snippet 1.

def compute_ndvi(df):
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df["ndvi"]

from robustraster import run
import glob as glob

files = glob.glob("./Plumas_Tiles/time_2018*.vrt")

run(
    dataset=files,
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles2",
        "report": True,
        "visualize_task_graph": "ndvi_task_graph.svg"
    },
    dask_mode="custom",
    dask_config={
        "n_workers": 3,
        "threads_per_worker": 1,
        "memory_limit": "3g",
    },
)

In [ ]:
# Code Snippet 3: Rather than pulling the Landsat data first with get_bands(), we use our ee.ImageCollection
#                 as our UDD, compute NDVI, and export the results to a parquet file to disk.

def compute_ndvi(df):
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df["ndvi"]

from robustraster import run

run(
    dataset=ic,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
     function_tuning_config={
        "tune_function": True
    },
    export_config={
        "mode": "vector",
        "file_format": "parquet",
        "output_folder": "Plumas_NDVI_Parquet",
        "report": True
    }
)

In [4]:
print(files)

['./Plumas_Tiles\\time_20180101T000000.vrt', './Plumas_Tiles\\time_20180201T000000.vrt', './Plumas_Tiles\\time_20180301T000000.vrt', './Plumas_Tiles\\time_20180401T000000.vrt', './Plumas_Tiles\\time_20180501T000000.vrt', './Plumas_Tiles\\time_20180601T000000.vrt', './Plumas_Tiles\\time_20180701T000000.vrt', './Plumas_Tiles\\time_20180801T000000.vrt', './Plumas_Tiles\\time_20180901T000000.vrt', './Plumas_Tiles\\time_20181001T000000.vrt', './Plumas_Tiles\\time_20181101T000000.vrt', './Plumas_Tiles\\time_20181201T000000.vrt']


In [ ]:
# Code Snippet 4: Using our ee.ImageCollection as our UDD again, we pass in an R syntaxed version of
#                 our NDVI function. We export the results to Google Cloud Storage.

from robustraster import run
import glob as glob

# My UDF. We are still computing NDVI, but using R syntax.
r_code = """\
compute_ndvi_r <- function(df) {
    df$ndvi <- (df$SR_B5 - df$SR_B4) / (df$SR_B5 + df$SR_B4)
    return(df[, c("time", "X", "Y", "ndvi")])
}
"""

# run(
#     dataset=ic,
#     dataset_config={
#         'geometry': Plumas_Boundaries,
#         'crs': 'EPSG:3310',
#         'scale': 30,
#     },
#     user_function_config={
#         "is_r_function": True,
#         "r_function_code": r_code,
#         "r_function_name": "compute_ndvi_r",
#     },
#     export_config={
#         "mode": "raster",
#         "file_format": "GTiff",
#         "output_folder": "Plumas_NDVI_Tiles_R_Docker_GCS",
#         "export_to_gcs": True,
#         "gcs_credentials": r"C:\Users\Adriano Matos\Downloads\admin-493800-d65989e136bf.json",
#         "gcs_bucket": "robustraster-bucket",

#     },
#     #docker_image="adrianomdocker/robustraster"
#     docker_image = "adrianomdocker/r042"
# )
# #path\to\service-account-credentials.jso

files = glob.glob("./Plumas_Tiles/time_2018*.vrt")

run(
    dataset=files,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "is_r_function": True,
        "r_function_code": r_code,
        "r_function_name": "compute_ndvi_r",
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles_R_Docker",
        "report": True
    },
    docker_image = "adrianomdocker/rr042"
)

In [ ]:
from robustraster import run

# My UDF. We are still computing NDVI, but using R syntax.
r_code = """\
compute_ndvi_r <- function(df) {
    df$ndvi <- (df$SR_B5 - df$SR_B4) / (df$SR_B5 + df$SR_B4)
    return(df[, c("time", "X", "Y", "ndvi")])
}
"""

run(
    dataset=ic,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "is_r_function": True,
        "r_function_code": r_code,
        "r_function_name": "compute_ndvi_r",
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles_R_Docker",
        "report": True
    },
    docker_image = "adrianomdocker/robustraster"
)

In [ ]:
# Upload to GEE example

def compute_ndvi(df):
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df["ndvi"]

from robustraster import run

run(
    dataset=ic,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
     function_tuning_config={
        "tune_function": True
    },
    export_config={
        "mode": "vector",
        "file_format": "parquet",
        "output_folder": "Plumas_NDVI_Parquet",
        "report": True
    }
)

In [ ]:
from robustraster import run

# 1. Define your custom user function
def compute_ndvi(df):
    df["ndvi"] = (df["SR_B5"] - df["SR_B4"]) / (df["SR_B5"] + df["SR_B4"])
    return df

# 3. Define a small Area of Interest (AOI) as a GEE Polygon geometry.
#    The coordinates describe a bounding box located in California's
#    Sierra Nevada region, specified in WGS84 (EPSG:4326) longitude/latitude.
#    The polygon is closed by repeating the first coordinate at the end.
aoi = ee.Geometry.Polygon(
    [[[-120.5, 38.5],     # Southwest corner (lower-left)
      [-120.5, 38.55],    # Northwest corner (upper-left)
      [-120.4, 38.55],    # Northeast corner (upper-right)
      [-120.4, 38.5],     # Southeast corner (lower-right)
      [-120.5, 38.5]]]    # Close the polygon at the starting vertex
)

# 2. Run the function
run(
    # The dataset to process (e.g., an Earth Engine ImageCollection)
    dataset=ic, 
    
    # Dataset configurations
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    
    # Your custom function
    user_function_config={
        "user_function": compute_ndvi,
    },
    
    # Export Configuration (Automated GEE Upload!)
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "output_tiles",
        
        # Google Cloud Storage Settings
        "export_to_gcs": True,
        "gcs_bucket": "rrbucket2345",
        "gcs_folder": "ndvi-2018-exports", # Optional subfolder
        "gcs_project": "project-459a1b61-31e0-47b6-aa9", # <--- Add this!
        
        # Google Earth Engine Upload Settings
        "upload_results_to_gee": True,
        "gee_asset_path": "projects/project-459a1b61-31e0-47b6-aa9/assets/ndvi"
    }
)


Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:57480' processes=16 threads=16, memory=29.59 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] AOI tiling enabled. Streaming tiles in batches...
[robustraster] Processing tile 1 of 1
[robustraster] ✅ Tile 1 already succeeded; skipping.


c:\anaconda3\envs\robustraster\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


[robustraster] Found 12 files across 12 time steps. Starting GEE ingestion...
[robustraster] Parent asset 'projects/project-459a1b61-31e0-47b6-aa9/assets/ndvi' may not exist. Attempting to create it as an ImageCollection...
[robustraster] ✅ Successfully created ImageCollection: projects/project-459a1b61-31e0-47b6-aa9/assets/ndvi
[robustraster] ✅ Earth Engine task 914b4213-5a4e-4521-9a3b-be7dce7ad975 launched for asset projects/project-459a1b61-31e0-47b6-aa9/assets/ndvi/2018_01_01. Tiles: 1.
[robustraster] ✅ Earth Engine task 529e1cf9-1750-4f96-b9aa-da9f80ef3d92 launched for asset projects/project-459a1b61-31e0-47b6-aa9/assets/ndvi/2018_02_01. Tiles: 1.
[robustraster] ✅ Earth Engine task 5e9f5fb8-47f5-4e99-8d4f-94e5e5080935 launched for asset projects/project-459a1b61-31e0-47b6-aa9/assets/ndvi/2018_03_01. Tiles: 1.
[robustraster] ✅ Earth Engine task 370d1ea5-1295-4421-8505-1599d9161cb7 launched for asset projects/project-459a1b61-31e0-47b6-aa9/assets/ndvi/2018_04_01. Tiles: 1.
[robustra